In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
import pandas as pd


In [23]:
# Set up Selenium WebDriver (e.g., ChromeDriver)
driver = webdriver.Chrome()

# Navigate to Qualtrics Login Page
driver.get('https://uofsc.pdx1.qualtrics.com/login')


In [ ]:
# Sign in
username_field = driver.find_element(By.ID, 'UserName')
password_field = driver.find_element(By.ID, 'UserPassword')
username_field.clear()
password_field.clear()
account = 'wedell@mailbox.sc.edu'
password = '1800Gervais'
username_field.send_keys(account)
password_field.send_keys(password)
password_field.send_keys(Keys.RETURN)


In [ ]:
# Wait for the page to load
time.sleep(5)

# Now, go to the page that lists your stimuli (or folders)
page_url = 'https://uofsc.pdx1.qualtrics.com/app/library#libraryID=UR_5uvSniYAMbURnkV&folderID=FLDR_h945BCYwIHhBa3D'
driver.get(f"{page_url}&page=1")

# Wait for the page to load
time.sleep(5)

# Use BeautifulSoup to extract how many pages there are under the target folder
soup = BeautifulSoup(driver.page_source, 'html.parser')

input_element  = soup.find_all('input', {'type':'number'})
min_value = int(input_element[0].get('min'))
max_value = int(input_element[0].get('max'))


df_url = []
for pageid in range(1,max_value+1):
    driver.get(f"{page_url}&page={pageid}")
    time.sleep(5)
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    List_stimuli = soup.find_all('tr', {'role':'row'})
    for elem in List_stimuli:
        df_tmp = pd.DataFrame({'label':elem.get('aria-label'),'url':elem.get('id')},index=[0])
        df_url.append(df_tmp)

    print(f"page: {pageid}/{max_value}")
df_url = pd.concat(df_url,ignore_index=True)


page: 1/4
page: 2/4
page: 3/4
page: 4/4


In [104]:
df_url

,label,url
0,Allison_AgedMother001.wav,F_T5SOkEh1QWDo8aq
1,Allison_AgedMother002.wav,F_O5AKm5XACDNm1TD
2,Allison_AgedMother003.wav,F_9qnUMwhAWqbLZha
3,Allison_AgedMother004.wav,F_OGicyWjSIR6aned
4,Allison_AgedMother005.wav,F_j61MItqk42heYw3
...,...,...
95,Allison_AgedMother029.wav,F_1U3DX85JrU5PkOI
96,Allison_AgedMother030.wav,F_JkYMAJmozw3TKu7
97,Allison_AgedMother031.wav,F_jKCtKjZXcXcpZqL
98,Allison_AgedMother032.wav,F_4lLmE5sROtMyHKA


In [105]:
driver.quit()